Import Panda to read the file

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import zscore
import matplotlib.pyplot as plt
import seaborn as sns

Read Benin csv file

In [ ]:
df = pd.read_csv('../data/benin-malanville.csv')

In [8]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
GHI,525600.0,240.559452,331.131327,-12.9,-2.0,1.8,483.4,1413.0
DNI,525600.0,167.187516,261.710501,-7.8,-0.5,-0.1,314.2,952.3
DHI,525600.0,115.358961,158.691074,-12.6,-2.1,1.6,216.3,759.2
ModA,525600.0,236.589496,326.894859,0.0,0.0,4.5,463.7,1342.3
ModB,525600.0,228.883576,316.536515,0.0,0.0,4.3,447.9,1342.3
Tamb,525600.0,28.179683,5.924297,11.0,24.2,28.0,32.3,43.8
RH,525600.0,54.487969,28.073069,2.1,28.8,55.1,80.1,100.0
WS,525600.0,2.121113,1.603466,0.0,1.0,1.9,3.1,19.5
WSgust,525600.0,2.809195,2.029120,0.0,1.3,2.6,4.1,26.6
WSstdev,525600.0,0.473390,0.273395,0.0,0.4,0.5,0.6,4.2


In [ ]:
missing = df.isna().sum().sort_values(ascending=False)
missing_pct = (missing / len(df) * 100).round(2)
cols_over5 = missing_pct[missing_pct > 5].index.tolist()
cols_over5

Timestamp             0
GHI                   0
DNI                   0
DHI                   0
ModA                  0
ModB                  0
Tamb                  0
RH                    0
WS                    0
WSgust                0
WSstdev               0
WD                    0
WDstdev               0
BP                    0
Cleaning              0
Precipitation         0
TModA                 0
TModB                 0
Comments         525600
dtype: int64

In [28]:
# cols = ['GHI','DNI','DHI','ModA','ModB','WS','WSgust']

# # Create z-score columns
# for col in cols:
#     df[col + '_zscore'] = zscore(df[col])

# # Find outliers where any z-score > 3 or < -3
# outliers = df[(df[[col + '_zscore' for col in cols]].abs() > 3).any(axis=1)]
# outliers.head()

cols = ["GHI","DNI","DHI","ModA","ModB","WS","WSgust"]
# compute zscore ignoring NaNs
z = df[cols].apply(lambda x: (x - x.mean())/x.std(ddof=0))
outlier_mask = (z.abs() > 3)
df_outliers = df[outlier_mask.any(axis=1)]
print("Outliers flagged:", len(df_outliers))

# add column marks for review
for c in cols:
    df[f"{c}_zflag"] = z[c].abs() > 3


# df_outliers.head()
df.head(20)



Outliers flagged: 7740


,Timestamp,GHI,DNI,DHI,ModA,ModB,Tamb,RH,WS,WSgust,...,ModB_zscore,WS_zscore,WSgust_zscore,GHI_zflag,DNI_zflag,DHI_zflag,ModA_zflag,ModB_zflag,WS_zflag,WSgust_zflag
0,2021-08-09 00:01,-1.2,-0.2,-1.1,0.0,0.0,26.2,93.4,0.0,0.4,...,-0.723088,-1.322831,-1.187312,False,False,False,False,False,False,False
1,2021-08-09 00:02,-1.1,-0.2,-1.1,0.0,0.0,26.2,93.6,0.0,0.0,...,-0.723088,-1.322831,-1.384442,False,False,False,False,False,False,False
2,2021-08-09 00:03,-1.1,-0.2,-1.1,0.0,0.0,26.2,93.7,0.3,1.1,...,-0.723088,-1.135736,-0.842334,False,False,False,False,False,False,False
3,2021-08-09 00:04,-1.1,-0.1,-1.0,0.0,0.0,26.2,93.3,0.2,0.7,...,-0.723088,-1.198101,-1.039464,False,False,False,False,False,False,False
4,2021-08-09 00:05,-1.0,-0.1,-1.0,0.0,0.0,26.2,93.3,0.1,0.7,...,-0.723088,-1.260466,-1.039464,False,False,False,False,False,False,False
5,2021-08-09 00:06,-1.0,-0.1,-1.0,0.0,0.0,26.2,93.8,0.0,0.4,...,-0.723088,-1.322831,-1.187312,False,False,False,False,False,False,False
6,2021-08-09 00:07,-1.0,-0.1,-1.0,0.0,0.0,26.2,93.7,0.0,0.0,...,-0.723088,-1.322831,-1.384442,False,False,False,False,False,False,False
7,2021-08-09 00:08,-1.0,-0.1,-1.0,0.0,0.0,26.2,93.7,0.7,1.3,...,-0.723088,-0.886277,-0.743769,False,False,False,False,False,False,False
8,2021-08-09 00:09,-1.0,-0.1,-1.0,0.0,0.0,26.2,93.6,0.4,1.1,...,-0.723088,-1.073371,-0.842334,False,False,False,False,False,False,False
9,2021-08-09 00:10,-1.0,-0.1,-1.0,0.0,0.0,26.2,93.6,0.5,1.1,...,-0.723088,-1.011006,-0.842334,False,False,False,False,False,False,False


In [29]:

key_cols = ["GHI","DNI","DHI","ModA","ModB","Tamb","RH","WS"]
df_clean = df.copy()
for c in key_cols:
    if df_clean[c].isna().any():
        df_clean[c] = df_clean[c].fillna(df_clean[c].median())
# for small time gaps: optionally forward fill
df_clean = df_clean.sort_index()
df_clean = df_clean.ffill(limit=3)  # forward-fill up to 3 periods
df_clean


,Timestamp,GHI,DNI,DHI,ModA,ModB,Tamb,RH,WS,WSgust,...,ModB_zscore,WS_zscore,WSgust_zscore,GHI_zflag,DNI_zflag,DHI_zflag,ModA_zflag,ModB_zflag,WS_zflag,WSgust_zflag
0,2021-08-09 00:01,-1.2,-0.2,-1.1,0.0,0.0,26.2,93.4,0.0,0.4,...,-0.723088,-1.322831,-1.187312,False,False,False,False,False,False,False
1,2021-08-09 00:02,-1.1,-0.2,-1.1,0.0,0.0,26.2,93.6,0.0,0.0,...,-0.723088,-1.322831,-1.384442,False,False,False,False,False,False,False
2,2021-08-09 00:03,-1.1,-0.2,-1.1,0.0,0.0,26.2,93.7,0.3,1.1,...,-0.723088,-1.135736,-0.842334,False,False,False,False,False,False,False
3,2021-08-09 00:04,-1.1,-0.1,-1.0,0.0,0.0,26.2,93.3,0.2,0.7,...,-0.723088,-1.198101,-1.039464,False,False,False,False,False,False,False
4,2021-08-09 00:05,-1.0,-0.1,-1.0,0.0,0.0,26.2,93.3,0.1,0.7,...,-0.723088,-1.260466,-1.039464,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
525595,2022-08-08 23:56,-5.5,-0.1,-5.9,0.0,0.0,23.1,98.3,0.3,1.1,...,-0.723088,-1.135736,-0.842334,False,False,False,False,False,False,False
525596,2022-08-08 23:57,-5.5,-0.1,-5.8,0.0,0.0,23.1,98.3,0.2,0.7,...,-0.723088,-1.198101,-1.039464,False,False,False,False,False,False,False
525597,2022-08-08 23:58,-5.5,-0.1,-5.8,0.0,0.0,23.1,98.4,0.6,1.1,...,-0.723088,-0.948641,-0.842334,False,False,False,False,False,False,False
525598,2022-08-08 23:59,-5.5,-0.1,-5.8,0.0,0.0,23.1,98.3,0.9,1.3,...,-0.723088,-0.761547,-0.743769,False,False,False,False,False,False,False


In [30]:
df_clean.to_csv('../data/benin_clean.csv', index=False)